# **Các thư viện cần thiết**

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder, MinMaxScaler, StandardScaler, scale
from sklearn.feature_selection import SelectKBest, f_classif, chi2
from sklearn.utils import resample
from imblearn.under_sampling import RandomUnderSampler, NearMiss, EditedNearestNeighbours
from itertools import cycle
import joblib
import pickle
# Ẩn các warnings
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# **CTU-13**

# **Đọc bộ dữ liệu CTU-13 từ Drive**

In [ ]:
# Dùng đường dẫn đến file CTU_13.csv
df_CTU_13 = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/CTU_13.csv', index_col=0)

In [ ]:
df_CTU_13

Số lượng các giá trị bị thiếu trong các cột/ tính năng

In [ ]:
df_CTU_13.isnull().sum()

# **Xoá các cột ‘StartTime’, ‘SrcAddr’, ‘Sport’, ‘DstAddr’, ‘Dport’, ‘State’**

In [ ]:
df_CTU_13 = df_CTU_13.drop(['StartTime','SrcAddr','Sport','DstAddr','Dport','State'], axis=1)
df_CTU_13

,Dur,Proto,Dir,sTos,dTos,TotPkts,TotBytes,SrcBytes,Label
0,1.026539,tcp,->,0.0,0.0,4,276,156,flow=Background-Established-cmpgw-CVUT
1,1.009595,tcp,->,0.0,0.0,4,276,156,flow=Background-Established-cmpgw-CVUT
2,3.056586,tcp,->,0.0,0.0,3,182,122,flow=Background-TCP-Attempt
3,3.111769,tcp,->,0.0,0.0,3,182,122,flow=Background-TCP-Attempt
4,3.083411,tcp,->,0.0,0.0,3,182,122,flow=Background-TCP-Attempt
...,...,...,...,...,...,...,...,...,...
19976695,0.000000,udp,->,0.0,NaN,1,476,476,flow=Background-UDP-Attempt
19976696,0.000427,udp,<->,0.0,0.0,2,135,75,flow=Background-UDP-Established
19976697,0.000000,udp,->,0.0,NaN,1,476,476,flow=Background-UDP-Attempt
19976698,0.000000,udp,->,0.0,NaN,1,476,476,flow=Background-UDP-Attempt


# **1. Xử lý các giá trị NaN**

In [ ]:
# Thay thế các giá trị NaN bằng số 9
df_CTU_13 = df_CTU_13.fillna(9)
df_CTU_13

In [ ]:
df_CTU_13.isnull().sum()

# **Gán lại nhãn mới cho cột Label**

*  Background traffic là 0
* Botnet traffic (DDoS traffic) là 1
*  Normal traffic là 2


In [ ]:
lst = []
for i in df_CTU_13['Label']:
    if 'Botnet' in i:
        lst.append(1)
    elif 'Normal' in i:
        lst.append(2)
    else:
        lst.append(0)

df_CTU_13['Label'] = lst
df_CTU_13

In [ ]:
df_CTU_13['Label'].value_counts()

# **2.1. Mã hoá với giá trị số**

Gán giá trị số cho các giá trị trong cột 'Proto'

In [ ]:
# udp = 17, tcp = 6, icmp = 1, other = 0
protocol_number = []
for i in df_CTU_13['Proto']:
    if i == 'udp':
        protocol_number.append(17)
    elif i == 'tcp':
        protocol_number.append(6)
    elif i == 'icmp':
        protocol_number.append(1)
    else:
        protocol_number.append(0)

df_CTU_13['Proto'] = protocol_number
df_CTU_13

Gán giá trị số cho các giá trị trong cột 'Dir'

In [ ]:
# '  <->' = 1, '   ->' = 2, other = 0
direction_number = []
for i in df_CTU_13['Dir']:
    if i == '  <->':
        direction_number.append(1)
    elif i == '   ->':
        direction_number.append(2)
    else:
        direction_number.append(0)

df_CTU_13['Dir'] = direction_number
df_CTU_13

In [ ]:
A = df_CTU_13.copy()

In [ ]:
X_new_df = A.drop(['Label'], axis=1).copy()
y = A['Label'].copy()

# **2.2. Mã hoá One-Hot Encoding**

In [ ]:
A = df_CTU_13.copy()

In [ ]:
def draw_plot(col):
    if col.name == 'sTos' or col.name == 'dTos':
        col = col.astype('str')
    names = col.value_counts().index.tolist()
    values = col.value_counts().tolist()
    fig, ax = plt.subplots(figsize =(16, 6)) 
    ax.bar(names, values, color='Blue')
    plot_name = col.name + ' values'
    ax.set_title(plot_name)
    for index, data in enumerate(values):
        plt.text(x=index, y=data+50000, s=f"{data}", fontdict=dict(fontsize=10), horizontalalignment='center')  
    plt.tight_layout()
    plt.show()

one_hot_lst = ['Dir','Proto','sTos','dTos']
for i in one_hot_lst:
    draw_plot(A[i])

In [ ]:
one_hot_lst = ['Dir','Proto','sTos','dTos']
column_list = ['Dur','TotPkts','TotBytes','SrcBytes',
               'Dir_   ->','Dir_  <->','Dir_others',
	             'Proto_icmp','Proto_tcp','Proto_udp','Proto_others',
               'sTos_0.0','sTos_9.0','sTos_others','dTos_0.0','dTos_9.0','dTos_others','Label']

def one_hot_encoding(df):
    df_new = pd.DataFrame()
    other_list = []
    for i in one_hot_lst:
        if i == 'Dir':
            for j in df[i]:
                if j == '  <->' or j == '   ->':
                    other_list.append(0)
                else:
                    other_list.append(1)
            other_name = 'Dir_others'
            df[other_name] = other_list
            other_list.clear()
            df_new = pd.get_dummies(df, columns=[i])
        elif i == 'Proto':
            for j in df[i]:
                if j == 'tcp' or j == 'udp' or j == 'icmp':
                    other_list.append(0)
                else:
                    other_list.append(1)
            other_name = 'Proto_others'
            df_new[other_name] = other_list
            other_list.clear()
            df_new = pd.get_dummies(df_new, columns=[i])
        elif i == 'sTos':
            for j in df[i]:
                if j == 0 or j == 9:
                    other_list.append(0)
                else:
                    other_list.append(1)
            other_name = 'sTos_others'
            df_new[other_name] = other_list
            other_list.clear()
            df_new = pd.get_dummies(df_new, columns=[i])
        elif i == 'dTos':
            for j in df[i]:
                if j == 0 or j == 9:
                    other_list.append(0)
                else:
                    other_list.append(1)
            other_name = 'dTos_others'
            df_new[other_name] = other_list
            other_list.clear()
            df_new = pd.get_dummies(df_new, columns=[i])
    for col in df_new.columns:
        if col not in column_list:
            df_new = df_new.drop([col], axis=1)
    return df_new

A = one_hot_encoding(A)
A

In [ ]:
# Tách thành tập X và y
X = A.drop(['Label'], axis=1).copy()
y = A['Label'].copy()

# **3.1. Chi-Squared**

In [ ]:
X_new = SelectKBest(chi2, k=10).fit_transform(X, y)
print(X_new.shape)
X_new

In [ ]:
X_new_df = pd.DataFrame(X_new)
X_new_df.describe()

So sánh bảng X_new_df.describe() với bảng X.describe(). Ta biết được 10 cột được giữ lại là:

'Dur', 'TotPkts', 'TotBytes', 'SrcBytes', 'Dir_1', 'Dir_2', 'Proto_icmp', 'Proto_tcp', 'Proto_udp', 'dTos_9.0'

# **3.2. ANOVA**

In [ ]:
X_new = SelectKBest(f_classif, k=10).fit_transform(X, y)
print(X_new.shape)
X_new

In [ ]:
X_new_df = pd.DataFrame(X_new)
X_new_df.describe()

So sánh bảng X_new_df.describe() với bảng X.describe(). Ta biết được 10 cột được giữ lại là:

'Dur', 'Dir_others', 'Dir_1', 'Dir_2', 'Proto_icmp', 'Proto_tcp', 'Proto_udp', 'sTos_0.0', 'dTos_0.0', 'dTos_9.0'

# **4.1. UnderSampling với RandomUnderSampler**

In [ ]:
rus = RandomUnderSampler(random_state=0, sampling_strategy={0: 801132, 1: 444699, 2: 356433})
X_res, y_res = rus.fit_resample(X_new_df, y)

In [ ]:
y_res.value_counts()

In [ ]:
X_res

# **4.2. UnderSampling với NearMiss**

NearMiss version 1

In [ ]:
nearmiss = NearMiss(version=1, sampling_strategy={0: 801132, 1: 444699, 2: 356433})
X_res, y_res = nearmiss.fit_resample(X_new_df, y)

In [ ]:
y_res.value_counts()

In [ ]:
X_res

# **Tách X_train, X_test, y_train, y_test**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, train_size=0.8, test_size=0.2, random_state=0)

In [ ]:
y_test.value_counts()

# **5.1. Normalization**

In [ ]:
# fit scaler on training data
scaler = MinMaxScaler(feature_range=(0,1)).fit(X_train)

# transform training data
X_train = scaler.transform(X_train)

# transform testing dataabs
X_test = scaler.transform(X_test)

# **5.2. Standardization**

In [ ]:
# fit scaler on training data
scaler = StandardScaler().fit(X_train)

# transform training data
X_train = scaler.transform(X_train)

# transform testing dataabs
X_test = scaler.transform(X_test)

# **Decision Tree**

In [ ]:
model_DT = DecisionTreeClassifier(random_state=0)
model_DT.fit(X_train, y_train)

In [ ]:
y_predict_DT = model_DT.predict(X_test)
acc_score_DT = accuracy_score(y_test, y_predict_DT)
print("Accuracy score: {} %".format(acc_score_DT*100))


# **Random Forest**

In [ ]:
model_RF = RandomForestClassifier(random_state=0)
model_RF.fit(X_train, y_train)

In [ ]:
y_predict_RF = model_RF.predict(X_test)
acc_score_RF = accuracy_score(y_test, y_predict_RF)
print("Accuracy score: {} %".format(acc_score_RF*100))


# **Naive Bayes**

In [ ]:
model_NB = GaussianNB()
model_NB.fit(X_train, y_train)

In [ ]:
y_predict_NB = model_NB.predict(X_test)
acc_score_NB = accuracy_score(y_test, y_predict_NB)
print("Accuracy score: {} %".format(acc_score_NB*100))


# **K-Nearest Neighbors**

In [ ]:
model_KNN = KNeighborsClassifier()
model_KNN.fit(X_train, y_train)

In [ ]:
y_predict_KNN = model_KNN.predict(X_test)
acc_score_KNN = accuracy_score(y_test, y_predict_KNN)
print("Accuracy score: {} %".format(acc_score_KNN*100))
